# Plan de medios con presupuesto — Optimización binaria con PuLP

**Problema:** escoger qué canales publicitarios usar este mes para **maximizar el impacto** sin exceder el presupuesto.

| Canal | Costo | Impacto |
|---|---|---|
| TV | 8 | 14 |
| Radio | 3 | 5 |
| Redes sociales | 4 | 7 |
| Prensa | 2 | 3 |

**Presupuesto disponible:** 9 · **Solución esperada:** impacto = 15

## 1. Formulación del modelo

Es un **problema de la mochila 0/1**: cada canal se contrata completo o no se contrata.

**Variables de decisión**

$$x_i = \begin{cases} 1 & \text{si se usa el canal } i \\ 0 & \text{si no} \end{cases} \qquad i \in \{\text{TV, Radio, Redes, Prensa}\}$$

**Función objetivo** (maximizar impacto)

$$\max Z = 14x_{TV} + 5x_{Radio} + 7x_{Redes} + 3x_{Prensa}$$

**Restricción de presupuesto**

$$8x_{TV} + 3x_{Radio} + 4x_{Redes} + 2x_{Prensa} \le 9$$

$$x_i \in \{0,1\}$$

In [1]:
# !pip install "pulp==3.3.2"   # PuLP 4.0 cambió la API; este notebook usa la sintaxis clásica (2.x/3.x)
import pulp
import pandas as pd

## 2. Datos

In [2]:
canales = ["TV", "Radio", "Redes sociales", "Prensa"]
costo   = {"TV": 8, "Radio": 3, "Redes sociales": 4, "Prensa": 2}
impacto = {"TV": 14, "Radio": 5, "Redes sociales": 7, "Prensa": 3}
presupuesto = 9

pd.DataFrame({"Costo": costo, "Impacto": impacto})

,Costo,Impacto
TV,8,14
Radio,3,5
Redes sociales,4,7
Prensa,2,3


## 3. Construcción del modelo en PuLP

In [3]:
modelo = pulp.LpProblem("Plan_de_medios", pulp.LpMaximize)

# Variables binarias
x = pulp.LpVariable.dicts("x", canales, cat=pulp.LpBinary)

# Función objetivo
modelo += pulp.lpSum(impacto[c] * x[c] for c in canales), "Impacto_total"

# Restricción de presupuesto
modelo += pulp.lpSum(costo[c] * x[c] for c in canales) <= presupuesto, "Presupuesto"

print(modelo)

Plan_de_medios:
MAXIMIZE
3*x_Prensa + 5*x_Radio + 7*x_Redes_sociales + 14*x_TV + 0.0
SUBJECT TO
Presupuesto: 2 x_Prensa + 3 x_Radio + 4 x_Redes_sociales + 8 x_TV <= 9

VARIABLES
0 <= x_Prensa <= 1 Integer
0 <= x_Radio <= 1 Integer
0 <= x_Redes_sociales <= 1 Integer
0 <= x_TV <= 1 Integer



## 4. Solución

In [4]:
modelo.solve(pulp.PULP_CBC_CMD(msg=False))

print("Estado:", pulp.LpStatus[modelo.status])
print("Impacto máximo:", pulp.value(modelo.objective))
print()
for c in canales:
    print(f"{c:15s} -> {'USAR' if x[c].value() == 1 else 'no usar'}")

Estado: Optimal
Impacto máximo: 15.0

TV              -> no usar
Radio           -> USAR
Redes sociales  -> USAR
Prensa          -> USAR


In [5]:
resultado = pd.DataFrame({
    "Canal": canales,
    "Costo": [costo[c] for c in canales],
    "Impacto": [impacto[c] for c in canales],
    "Seleccionado": [int(x[c].value()) for c in canales],
})
costo_usado = (resultado.Costo * resultado.Seleccionado).sum()
print(f"Costo usado: {costo_usado} de {presupuesto}")
resultado

Costo usado: 9 de 9


,Canal,Costo,Impacto,Seleccionado
0,TV,8,14,0
1,Radio,3,5,1
2,Redes sociales,4,7,1
3,Prensa,2,3,1


## 5. Verificación por fuerza bruta

Con 4 canales solo hay $2^4 = 16$ combinaciones, así que podemos comprobar que PuLP encontró el óptimo.

In [6]:
from itertools import product

combinaciones = []
for sel in product([0, 1], repeat=len(canales)):
    c_tot = sum(costo[c] * s for c, s in zip(canales, sel))
    i_tot = sum(impacto[c] * s for c, s in zip(canales, sel))
    if c_tot <= presupuesto:
        combinaciones.append({"Canales": ", ".join(c for c, s in zip(canales, sel) if s) or "(ninguno)",
                              "Costo": c_tot, "Impacto": i_tot})

pd.DataFrame(combinaciones).sort_values("Impacto", ascending=False).reset_index(drop=True)

,Canales,Costo,Impacto
0,"Radio, Redes sociales, Prensa",9,15
1,TV,8,14
2,"Radio, Redes sociales",7,12
3,"Redes sociales, Prensa",6,10
4,"Radio, Prensa",5,8
5,Redes sociales,4,7
6,Radio,3,5
7,Prensa,2,3
8,(ninguno),0,0


## 6. Interpretación

La solución óptima es usar **Radio + Redes sociales + Prensa**: cuesta exactamente 9 (todo el presupuesto) y genera un **impacto de 15**.

Aunque TV tiene el mayor impacto individual (14), consume 8 de las 9 unidades de presupuesto y no deja espacio para ningún otro canal. Combinar tres canales más baratos da más impacto total — por eso un criterio voraz ("elige el canal de mayor impacto primero") falla y conviene resolverlo como programa entero.

## 7. Función reutilizable (la misma lógica que usa la app de Streamlit)

In [7]:
def resolver_plan(costo: dict, impacto: dict, presupuesto: float):
    canales = list(costo)
    m = pulp.LpProblem("Plan_de_medios", pulp.LpMaximize)
    x = pulp.LpVariable.dicts("x", canales, cat=pulp.LpBinary)
    m += pulp.lpSum(impacto[c] * x[c] for c in canales)
    m += pulp.lpSum(costo[c] * x[c] for c in canales) <= presupuesto
    m.solve(pulp.PULP_CBC_CMD(msg=False))
    elegidos = [c for c in canales if x[c].value() == 1]
    return pulp.value(m.objective), elegidos

# Análisis de sensibilidad: impacto óptimo según el presupuesto
pd.DataFrame([
    {"Presupuesto": p, "Impacto": resolver_plan(costo, impacto, p)[0] or 0,
     "Canales": ", ".join(resolver_plan(costo, impacto, p)[1])}
    for p in range(0, 18)
])

,Presupuesto,Impacto,Canales
0,0,0.0,
1,1,0.0,
2,2,3.0,Prensa
3,3,5.0,Radio
4,4,7.0,Redes sociales
5,5,8.0,"Radio, Prensa"
6,6,10.0,"Redes sociales, Prensa"
7,7,12.0,"Radio, Redes sociales"
8,8,14.0,TV
9,9,15.0,"Radio, Redes sociales, Prensa"
